# Lesson 02 — The Three Main Types of Machine Learning\n\n## What this notebook does\n\nThis notebook builds three tiny, from-scratch demos — one for each of the three ways a machine can learn: **supervised**, **unsupervised**, and **reinforcement** learning. All three demos reuse the same small world of customer-support tickets, so you can compare the three approaches side by side on familiar ground. No external libraries are used — only Python's standard library (`collections.Counter` and `itertools`)."

## Part 1 — Supervised learning: a categorizer that studies labeled tickets\n\nWe give the code six support tickets, each already tagged with its true category (`billing`, `shipping`, or `account`) — that tag is called a **label**. The code counts which words tend to show up in each category's labeled tickets. Later, to guess a brand-new ticket's category, it just adds up how much each of its words leaned toward each label during training, and picks the highest-scoring label. This is supervised learning end to end: learn from already-answered examples, then answer new ones the same way.

In [ ]:
from collections import Counter

labeled_tickets = [
    ("My card was charged twice for one order", "billing"),
    ("I was billed for an item I never received", "billing"),
    ("When will my package arrive", "shipping"),
    ("The tracking number you gave me does not work", "shipping"),
    ("I can not log into my account", "account"),
    ("Please reset my password, it is not working", "account"),
]


def tokenize(text):
    return text.lower().replace(",", "").replace(".", "").split()


def train_category_word_counts(labeled_examples):
    counts_by_category = {}                             # one word-tally per category
    for text, category in labeled_examples:
        counts_by_category.setdefault(category, Counter())  # first time: start an empty tally
        counts_by_category[category].update(tokenize(text))  # add this ticket's words to it
    return counts_by_category


def predict_category(text, counts_by_category):
    words = tokenize(text)

    # Score each category: how much do this ticket's words lean toward it?
    scores = {}                                        # one score per category
    for category, counts in counts_by_category.items():
        category_score = 0                             # start this category at zero
        for word in words:                             # add its tally for each word
            category_score += counts.get(word, 0)
        scores[category] = category_score

    # Pick the category with the highest score (plain loop, no max(key=...)).
    best_category = None
    best_score = None
    for category, category_score in scores.items():
        if best_score is None or category_score > best_score:
            best_score = category_score                # new best found
            best_category = category
    return best_category, scores


# Compact alternative (same result) - the two loops inside predict_category,
# written the idiomatic way. We keep the loops above as the version that runs:
#   scores = {category: sum(counts.get(w, 0) for w in words)
#             for category, counts in counts_by_category.items()}
#   best_category = max(scores, key=scores.get)
# A dict comprehension builds the scores dict in one line; max(..., key=scores.get)
# returns the category whose score is largest.

category_word_counts = train_category_word_counts(labeled_tickets)

Now we test the trained categorizer on two brand-new tickets it has never seen, and print the score it gives each possible category — not just its final guess — so we can see its reasoning.

In [ ]:
new_tickets = [
    "My refund never showed up on my card",
    "Where is my order, it has not arrived yet",
]

for ticket in new_tickets:
    category, scores = predict_category(ticket, category_word_counts)
    print(ticket)
    print(f"  predicted category: {category}  (scores: {scores})")

## Part 2 — Unsupervised learning: finding groups without being told the categories\n\nNow forget the labels completely — pretend nobody ever tagged these six tickets. The code below only looks at which pairs of tickets **share the same words**, and calls two tickets \"the same kind of thing\" whenever they share at least two words. Nothing here is told what \"billing\" or \"shipping\" means; any grouping that emerges comes purely from the raw text, which is the core idea of unsupervised learning — finding structure with no answer key at all.

In [ ]:
from itertools import combinations

# Pull just the ticket text out of each (text, label) pair, ignoring the label.
tickets_no_labels = []                                 # will hold the six ticket strings
for text, label in labeled_tickets:                    # go through every labeled pair
    tickets_no_labels.append(text)                     # keep only the text

# Compact alternative (same result):
#   tickets_no_labels = [text for text, _ in labeled_tickets]


def shared_word_count(text_a, text_b):
    words_a = set(tokenize(text_a))                    # unique words in ticket A
    words_b = set(tokenize(text_b))                    # unique words in ticket B
    shared = words_a & words_b                         # words in BOTH (set intersection)
    return len(shared)                                 # how many they share


# Compare every pair of tickets. Plain nested loops: i is the first ticket,
# j is a later ticket, so each pair is checked exactly once.
n = len(tickets_no_labels)
for i in range(n):                                     # first ticket in the pair
    for j in range(i + 1, n):                          # second ticket, always after i
        overlap = shared_word_count(tickets_no_labels[i], tickets_no_labels[j])
        if overlap >= 2:                               # "same kind" if they share 2+ words
            print(f"Ticket {i} and Ticket {j} share {overlap} words -> grouped together")
            print(f"  {tickets_no_labels[i]}")
            print(f"  {tickets_no_labels[j]}")

# Compact alternative (same pairing) - itertools.combinations yields each i<j pair:
#   for i, j in combinations(range(len(tickets_no_labels)), 2):

## Part 3 — Reinforcement learning: learning which vending machine pays off

Here there is no labeled data and no fixed text to group. Instead, an **agent** (the decision-maker — here, a small loop of code) repeatedly chooses between two vending machines, `A` and `B`. Each time it picks one, the **environment** (the machine) hands back a **reward**: `1` for a treat, `0` for nothing. The agent never sees the machines' true odds — only its own running average reward for each machine so far — and it always tries whichever machine currently *looks* best. That is reinforcement learning: learning which actions pay off purely from trial, reward, and error, with no dataset handed over in advance.

To keep this walkthrough exact and reproducible, the treats each machine would hand back, in order, are fixed in a list below, instead of using real randomness.

In [ ]:
from itertools import cycle

# The fixed treat sequences each machine hands back, looping forever.
machine_a_outcomes = cycle([0, 0, 1, 0])
machine_b_outcomes = cycle([1, 1, 0, 1, 1, 0, 1])


def pull_lever(machine):
    if machine == "A":                                 # pick the right machine's stream
        return next(machine_a_outcomes)                # next treat from A
    else:
        return next(machine_b_outcomes)                # next treat from B

# Compact alternative (same result):
#   return next(machine_a_outcomes) if machine == "A" else next(machine_b_outcomes)


results = {"A": [], "B": []}                           # rewards collected from each machine
trials = 8

for trial in range(1, trials + 1):
    # Work out each machine's average reward so far (0.5 if never tried yet).
    average_so_far = {}
    for machine in ("A", "B"):
        rewards = results[machine]                     # rewards seen from this machine
        if rewards:                                    # tried at least once?
            average_so_far[machine] = sum(rewards) / len(rewards)
        else:
            average_so_far[machine] = 0.5              # no data yet: assume 50/50

    # Choose the machine that currently looks best (plain loop, no max(key=...)).
    choice = None
    best_average = None
    for machine, average in average_so_far.items():
        if best_average is None or average > best_average:
            best_average = average                     # new best-looking machine
            choice = machine

    reward = pull_lever(choice)                        # actually pull that machine
    results[choice].append(reward)                     # remember what we got
    print(f"Trial {trial}: tried machine {choice}, got reward {reward}")

# Compact alternative for the two blocks inside the loop (same result):
#   average_so_far = {m: (sum(results[m]) / len(results[m])) if results[m] else 0.5
#                     for m in ("A", "B")}
#   choice = max(average_so_far, key=average_so_far.get)

print()
for machine in ("A", "B"):
    pulls = results[machine]
    if pulls:                                          # only report machines we actually tried
        average_reward = sum(pulls) / len(pulls)
        print(f"Machine {machine}: tried {len(pulls)} times, average reward {average_reward:.2f}")
    else:
        print(f"Machine {machine}: never tried")

## Comparing the three

- **Supervised learning** (Part 1) needed labels up front, and used them to score new, unseen text.
- **Unsupervised learning** (Part 2) had no labels at all, and found only *some* of the true groupings — it correctly paired the two billing tickets and the two account tickets, but missed the two shipping tickets entirely, because they happen to share no words with each other. Structure found this way is only as good as the surface signal (here, shared words) actually reveals.
- **Reinforcement learning** (Part 3) had no text and no labels — only actions and rewards, discovered by repeated trial and error over time.

Three completely different situations, three different types of learning. Picking the right one starts with asking: do I have labeled answers, do I have unlabeled data I want structure from, or do I only have an agent that can act and get feedback?